<div style="background: linear-gradient(120deg, #071E3D 0%, #173B67 58%, #426B91 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src="Figures/iteso.jpg" style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0;"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.5); padding-left: 28px; color: white;">
        <h1 style="margin: 0 0 8px 0; font-size: 1.5em; line-height: 1.3;">Proyecto Final Equipo 4</h1>
        <h3 style="margin: 0; font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>

# Guía e instrucciones

En este proyecto, cada equipo resolverá una pregunta relacionada con sus datos mediante un proceso reproducible de extracción, comprensión, limpieza, transformación y análisis. El trabajo debe integrar las herramientas estudiadas en los **Módulos I, II y III**, seleccionar las técnicas pertinentes y explicar por qué se eligieron.

El conjunto de datos y su liga se asignarán a cada equipo. 

**Importante:** no se espera aplicar mecánicamente todos los métodos a cualquier conjunto de datos. El equipo debe justificar la pertinencia de cada transformación. 

## Ficha del equipo y del conjunto de datos

Completa esta ficha antes de comenzar y actualízala si cambia la fuente o el alcance.

| Campo | Información del equipo |
|---|---|
| Número o nombre del equipo | `Equipo 4` |
| Nombre completo de integrante 1 | `Oscar Eduardo de la Fuente Chávez` |
| Nombre completo de integrante 2 | `Maria Rosario Duran Chávez` |
| Nombre completo de integrante 3 | `Diego Cristhofer Maria Mejia ` |
| Integrante responsable de coordinación | `Maria Rosario Duran Chávez` |
| **Liga del dataset asignado al equipo** |[Dataset](https://iteso01-my.sharepoint.com/:f:/r/personal/gdesirena_iteso_mx/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fgdesirena%5Fiteso%5Fmx%2FDocuments%2FITESO%2FIngenier%C3%ADa%20y%20Ciencia%20de%20Datos%2FLaboratorio%5Fde%5FProcesamiento%5Fde%5FDatos%5F02026%2FProyecto%20Final%2Fdata%2FDataProyectosFinales%2Fdiabetes%2B130%2Dus%2Bhospitals%2Bfor%2Byears%2B1999%2D2008&share=cgoXIRfBuBvlSoQ2l4hA%5FXRiEgUC5G36sUHk2iT0UqhuUNL5Jw)|

**Responsabilidades del equipo:** acuerden quién coordinara las asesorías sobre el proyecto, esta persona será la responsable de coordinar al equipo y al quién me estaré dirigiendo para preguntas sobre el proyecto.

# Notas
- Terminar de leer el artículo https://onlinelibrary.wiley.com/doi/10.1155/2014/781670
- Para el proyecto asumimos que los datos faltantes siguen un mecanismo MAR, porque en las gráficas y comparaciones realizadas no se observan patrones claros de ausencia. Esta es una suposición de trabajo; con los datos disponibles no se puede demostrar el mecanismo con certeza.
- Despivotizar los medicamentos (Nuevo)


In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import Markdown, display
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

## Etapa 1. Planteamiento del problema y objetivo

Antes de programar, escriban en lenguaje claro:

1. ¿Qué situación o necesidad aborda el proyecto?
2. ¿A quién afecta y por qué importa analizarla?
3. ¿Qué preguntas concretas podrían responder con el dataset?
4. ¿Qué evidencia permitiría responderla?
6. ¿Qué no puede concluirse a partir de estos datos?

Si hay una variable objetivo, documenten su definición, unidad de análisis, clases y horizonte temporal. Verifiquen que las variables predictoras estén disponibles en el momento en que se pretende realizar una predicción. Si el proyecto es descriptivo y no tiene objetivo, no inventen uno para poder aplicar WOE o LDA.

## Etapa 2. Fuente y extracción de datos: 

Elijan el método de lectura de acuerdo con el formato disponible, por ejemplo CSV, Excel, JSON, XML, texto, Parquet, HTML o datos geográficos e imágenes cuando sean pertinentes. No conviertan archivos a otro formato sin explicar el motivo. Guarden los datos originales sin sobrescribirlos y registren los pasos de extracción.

En el notebook incluyan el nombre del archivo, la ruta relativa utilizada y una inspección inicial: dimensiones, columnas, tipos, primeras filas y muestra de valores. Eviten rutas absolutas personales para que otras personas del curso puedan reproducir el trabajo.

#### 1. ¿Qué situación o necesidad aborda el proyecto?

Muchos pacientes diabéticos hospitalizados no reciben el manejo preventivo y terapéutico que la evidencia respalda, en parte porque el control de la glucosa en el hospital es poco sistemático. Cuando el manejo falla, el paciente puede volver a ingresar poco después del alta. El proyecto busca identificar, a partir de lo ocurrido durante la hospitalización, qué pacientes tienen más riesgo de ser readmitidos en menos de 30 días.

#### 2. ¿A quién afecta y por qué importa?
Pacientes: Una readmisión temprana suele indicar complicaciones, y afecta su morbilidad y mortalidad.

Hospitales y sistemas de salud: Las readmisiones aumentan los costos y suelen usarse como indicador de calidad de la atención.

Personal clínico: Saber qué perfiles tienen más riesgo permite enfocar el seguimiento posterior al alta (citas, educación, ajuste de tratamiento).

#### 3. Preguntas concretas que se podrían responder
¿Qué proporción de los ingresados termina en readmisión a menos de 30 días?
¿Cambia ese porcentaje según edad, tipo de admisión, especialidad del médico o tiempo de estancia?
¿Que variables se asocian con menos readmisiones?
¿Influyen las visitas previas (ambulatorias, de urgencias, hospitalizaciones) en el riesgo?
¿Qué tan bien se puede predecir la readmisión temprana con la información disponible al momento del alta?

#### 4. ¿Qué evidencia permitiría responderlas?
Descriptiva: Tasas de readmisión por grupo, con tablas y gráficos.
Asociativa: Pruebas de independencia (chi-cuadrado) y comparaciones de distribuciones entre readmitidos y no readmitidos.
Predictiva: Un modelo evaluado en datos no vistos, con métricas apropiadas para clases desbalanceadas (AUC, recall, precisión, no solo exactitud), porque la readmisión < a 30 días es minoritaria.

#### 5. ¿Qué no puede concluirse con estos datos?
Causalidad. Es un estudio observacional. Que el HbA1c medido se asocie con menos readmisiones no prueba que medirlo las cause: los pacientes más graves reciben tratamientos distintos.

Readmisiones en otros hospitales. Solo se ve lo que ocurre dentro de estos sistemas. Un paciente reingresado en otro hospital aparece como "no readmitido".

Vigencia actual. Los datos son de 1999-2008; la práctica clínica, los fármacos y los registros han cambiado.

Todos los pacientes diabéticos. Solo incluye ingresos de 1 a 14 días, con pruebas de laboratorio y medicación. Quedan fuera urgencias sin ingreso, estancias largas y pacientes sin laboratorios.

Equidad entre grupos. Las variables de edad, género y raza son sensibles. Los patrones observados no deben usarse para decidir sobre personas, y la raza puede reflejar desigualdades de acceso, no biología.

In [ ]:

dir_base=os.getcwd()
ruta = os.path.join(dir_base, 'diabetic_data.json')
ruta

In [ ]:
# df_original = datos tal como vienen del archivo. NO se modifica en todo el notebook.
# df = copia de trabajo: todas las transformaciones se hacen sobre ella (y luego sobre df_imputado).
df_original = pd.read_json(ruta)
df = df_original.copy()
print(df.head())
df.shape
print(df.columns)

In [ ]:
#Reemplazar ? con NaN
df = df.replace("?", np.nan)
df.isna().sum()

In [ ]:
#Tipos de columnas
print(df.dtypes)
df.nunique().sort_values()

In [ ]:
df["diag_1"].value_counts()

### Inspección inicial:
##### -Dimensiones:
101766 filas, 50 columnas

##### -Columnas: 
'encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

## Etapa 3. Comprensión y calidad de datos:

Preparen un diccionario de datos que incluya nombre de variable, descripción, tipo, unidad o categorías y papel dentro del análisis. Diferencien tipos físicos (entero, decimal, texto, fecha) de tipos conceptuales (continua, discreta, nominal, ordinal, identificador y objetivo).

Revisen, cuantifiquen y documenten:

- Registros y columnas, filas duplicadas y claves.
- Valores faltantes por variable y posibles patrones de ausencia.
- Valores fuera de rango, inconsistencias, unidades y formatos.
- Distribuciones, categorías raras, desequilibrio de clases y posibles valores atípicos.
- Variables que podrían filtrar información futura o identificar directamente a una persona.

Cada corrección debe tener una razón trazable. Conserven una copia de los datos originales y describan cuántas filas o valores cambian en cada paso. 

Si hay predicción, separen entrenamiento y prueba antes de aprender imputaciones, límites, categorías, bins o codificaciones que dependen de la muestra. 

In [ ]:
def clasificar(serie, umbral_discreta=20):
    #si una variable entera tiene 20 valores distintos o menos, es discreta.
    n = serie.nunique()
    prop = n / len(serie)

    if pd.api.types.is_numeric_dtype(serie):
        # 1. Identificador: casi cada fila tiene un valor distinto
        if prop > 0.95:
            return "identificador"
        # 2. Entero (o decimal que en realidad son enteros) -> conteo
        es_entero = (serie.dropna() % 1 == 0).all()
        if es_entero:
            return "discreta"
        # 3. Tiene decimales -> medición
        return "continua"

    return "categórica (nominal u ordinal: revisar)"

In [ ]:
tipos = {col: clasificar(df[col]) for col in df.columns}

tabla = pd.DataFrame.from_dict(tipos, orient="index", columns=["tipo_conceptual"])
tabla["Valores unicos"]=df.nunique()
tabla.loc["patient_nbr", "tipo_conceptual"] = "identificador"
tabla.loc["race", "tipo_conceptual"] = "categórica nominal"
#identifica al paciente y se repite entre ingresos, por eso la regla automática no lo detectó.
#Es el número que identifica a cada paciente, no describe ninguna característica suya
#tabla

In [ ]:
identificadores = ["encounter_id", "patient_nbr"]
objetivo = ["readmitted"]

ordinales = ["age", "weight", "max_glu_serum", "A1Cresult"]

discretas = ["time_in_hospital", "num_lab_procedures", "num_procedures",
             "num_medications", "number_outpatient", "number_emergency",
             "number_inpatient", "number_diagnoses"]

medicamentos = ["metformin", "repaglinide", "nateglinide", "chlorpropamide",
                "glimepiride", "acetohexamide", "glipizide", "glyburide",
                "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
                "miglitol", "troglitazone", "tolazamide", "insulin",
                "glyburide-metformin", "glipizide-metformin",
                "glimepiride-pioglitazone", "metformin-rosiglitazone",
                "metformin-pioglitazone"]

constantes = ["examide", "citoglipton"]   # un solo valor: se excluyen

# medical_specialty se elimina por su alto porcentaje de faltantes (52%).
nominales = (["race", "gender", "payer_code",
              "diag_1", "diag_2", "diag_3",
              "admission_type_id", "admission_source_id", "discharge_disposition_id",
              "change", "diabetesMed"]
             + medicamentos)


In [ ]:
enteras = ["admission_type_id", "discharge_disposition_id", "admission_source_id",
           "time_in_hospital", "num_lab_procedures", "num_procedures",
           "num_medications", "number_outpatient", "number_emergency",
           "number_inpatient", "number_diagnoses"]
df[enteras] = df[enteras].astype("Int64")

In [ ]:
tabla.loc[identificadores, "tipo_conceptual"] = "identificador"
tabla.loc[objetivo, "tipo_conceptual"] = "objetivo"
tabla.loc[ordinales, "tipo_conceptual"] = "ordinal"
tabla.loc[nominales, "tipo_conceptual"] = "nominal"
tabla.loc[constantes, "tipo_conceptual"] = "constante"
tabla.loc[discretas, "tipo_conceptual"] = "discreta"
#tabla

In [ ]:
descripciones = {
    # Identificadores y objetivo
    "encounter_id": "Número único de cada ingreso hospitalario.",
    "patient_nbr": "Número único de cada paciente. Se repite si el paciente tuvo varios ingresos.",
    "readmitted": "Si el paciente volvió al hospital tras el alta: en menos de 30 días (<30), después de 30 días (>30) o nunca (NO).",

    # Demográficas
    "race": "Raza o etnia reportada del paciente.",
    "gender": "Género del paciente.",
    "age": "Grupo de edad del paciente, en intervalos de 10 años.",
    "weight": "Rango de peso del paciente. Casi todos los valores faltan.",

    # Datos administrativos
    "admission_type_id": "Código del tipo de admisión (emergencia, urgente, electiva, etc.).",
    "admission_source_id": "Código del origen del paciente al ingresar (urgencias, derivación médica, otro hospital, etc.).",
    "discharge_disposition_id": "Código del destino del paciente al alta (casa, otro centro, fallecido, cuidados paliativos, etc.).",
    "payer_code": "Código del tipo de seguro o entidad que paga la atención.",
    "medical_specialty": "Especialidad médica del profesional que admitió al paciente.",

    # Estancia
    "time_in_hospital": "Días de hospitalización en este ingreso (de 1 a 14).",
    "num_lab_procedures": "Número de pruebas de laboratorio realizadas durante el ingreso.",
    "num_procedures": "Número de procedimientos (distintos de laboratorio) realizados durante el ingreso.",
    "num_medications": "Número de medicamentos distintos administrados durante el ingreso.",
    "number_diagnoses": "Número de diagnósticos registrados en el ingreso.",

    # Historial del año previo
    "number_outpatient": "Visitas ambulatorias del paciente en el año anterior al ingreso.",
    "number_emergency": "Visitas a urgencias del paciente en el año anterior al ingreso.",
    "number_inpatient": "Hospitalizaciones del paciente en el año anterior al ingreso.",

    # Diagnósticos y pruebas
    "diag_1": "Diagnóstico principal del ingreso, en código CIE-9.",
    "diag_2": "Primer diagnóstico secundario del ingreso, en código CIE-9.",
    "diag_3": "Segundo diagnóstico secundario del ingreso, en código CIE-9.",
    "max_glu_serum": "Resultado de la prueba de glucosa en suero (None = no se midió).",
    "A1Cresult": "Resultado de la prueba de HbA1c, que refleja el control de glucosa de los últimos meses (None = no se midió).",

    # Medicación general
    "change": "Si hubo algún cambio en la medicación de la diabetes durante el ingreso.",
    "diabetesMed": "Si se recetó algún medicamento para la diabetes durante el ingreso.",
}

# Medicamentos: misma descripción para todos
medicamentos_desc = {
    "examide": "sin variación (siempre No)",
    "citoglipton": "sin variación (siempre No)",
}
for med in medicamentos + constantes:
    extra = f" Columna {medicamentos_desc[med]}." if med in medicamentos_desc else ""
    descripciones[med] = (f"Si se administró {med} y cómo cambió la dosis durante el ingreso "
                          f"(No, Down, Steady, Up).{extra}")

tabla["descripcion"] = pd.Series(descripciones)
#tabla

In [ ]:
def tipo_fisico(serie):
    if pd.api.types.is_bool_dtype(serie):
        return "booleano"
    if pd.api.types.is_integer_dtype(serie):
        return "entero"
    if pd.api.types.is_float_dtype(serie):
        return "decimal"
    if pd.api.types.is_datetime64_any_dtype(serie):
        return "fecha"
    return "texto"

tabla["tipo_fisico"] = [tipo_fisico(df[c]) for c in tabla.index]
#tabla

In [ ]:
# 1. Unidades de las numéricas
unidades = {
    "time_in_hospital": "días",
    "num_lab_procedures": "pruebas (conteo)",
    "num_procedures": "procedimientos (conteo)",
    "num_medications": "medicamentos (conteo)",
    "number_diagnoses": "diagnósticos (conteo)",
    "number_outpatient": "visitas (conteo)",
    "number_emergency": "visitas (conteo)",
    "number_inpatient": "hospitalizaciones (conteo)",
}

# 2. Categorías: la lista si son pocas, o cuántas hay si son muchas
def categorias(serie, max_cat=10):
    vals = serie.dropna().unique()
    if len(vals) <= max_cat:
        texto = []
        for v in vals:
            texto.append(str(v))           # cada valor como texto
        texto.sort()                       # ordenar alfabéticamente
        return ", ".join(texto) # unir con comas
    return f"{len(vals)} categorías"

# 3. Unir todo
def unidad_o_categorias(col):
    if col in identificadores:
        return "Sin unidad"
    if col in unidades:
        return f"{unidades[col]} (rango {df[col].min()} a {df[col].max()})"
    return categorias(df[col])

tabla["unidad_o_categorias"] = [unidad_o_categorias(c) for c in tabla.index]

In [ ]:
#Papel dentro del dataset
tabla["papel"] = "predictora"
tabla.loc[identificadores, "papel"] = "identificador"
tabla.loc[objetivo, "papel"] = "objetivo"
tabla.loc[constantes, "papel"] = "excluir (un solo valor)"
tabla.loc["weight", "papel"] = "candidata a excluir (muchos faltantes)"
#tabla

In [ ]:
tabla.to_csv("diccionario_datos.csv", index_label="variable", encoding="utf-8-sig")

In [ ]:
tabla_md = tabla.reset_index().rename(columns={"index": "variable"})
#display(Markdown(tabla_md.to_markdown(index=False)))

Revisen, cuantifiquen y documenten:

- Registros y columnas, filas duplicadas y claves.
- Valores faltantes por variable y posibles patrones de ausencia.
- Valores fuera de rango, inconsistencias, unidades y formatos.
- Distribuciones, categorías raras, desequilibrio de clases y posibles valores atípicos.
- Variables que podrían filtrar información futura o identificar directamente a una persona.

In [ ]:
print(df.shape)                                 
print("Filas duplicadas completas:", df.duplicated().sum())
print("encounter_id repetidos:", df["encounter_id"].duplicated().sum())
print("Pacientes distintos:", df["patient_nbr"].nunique())
print("Pacientes con más de un ingreso:", (df["patient_nbr"].value_counts() > 1).sum())

In [ ]:
nulos = pd.DataFrame({
    "n_nulos": df.isnull().sum(),
    "pct_nulos": (df.isnull().mean() * 100).round(2),
})
nulos

#### Vacíos "ocultos" (valores que significan "sin dato" pero no son `?` ni NaN)

Además del `?` ya reemplazado, el archivo `IDS_mapping.xml` define códigos cuya descripción es *NULL*, *Not Available*, *Not Mapped* o *Unknown/Invalid*. Son faltantes con otro nombre. También `gender` trae 3 registros con `Unknown/Invalid`.

**No** se tratan como faltantes `None` en `max_glu_serum` y `A1Cresult`: según el diccionario significa *prueba no realizada*, que es información clínica real (en el artículo es la categoría "no medido" de HbA1c).

In [ ]:
# 1. Búsqueda general de textos que suenan a "vacío" en columnas de texto
import re
patron = re.compile(r"^\s*$|^(\?|na|n/a|nan|null|none|unknown|unknown/invalid|not available|not mapped|missing|-|--)$", re.I)
hallazgos = {}
for col in df.select_dtypes(exclude="number").columns:
    vals = df[col].dropna().astype(str)
    hits = vals[vals.map(lambda x: bool(patron.match(x)))].value_counts()
    if len(hits):
        hallazgos[col] = hits.to_dict()
print("Textos sospechosos por columna:", hallazgos)

# 2. Códigos numéricos que, según IDS_mapping.xml, significan "sin dato"
ocultos = {
    "gender": ["Unknown/Invalid"],
    "admission_type_id": [5, 6, 8],              # Not Available, NULL, Not Mapped
    "discharge_disposition_id": [18, 25, 26],    # NULL, Not Mapped, Unknown/Invalid
    "admission_source_id": [9, 15, 17, 20, 21],  # Not Available, Not Available, NULL, Not Mapped, Unknown/Invalid
}

resumen = []
for col, valores in ocultos.items():
    n_antes = df[col].isna().sum()
    n_ocultos = df[col].isin(valores).sum()
    df[col] = df[col].mask(df[col].isin(valores))      # se convierten en NaN
    resumen.append({"columna": col, "nulos_antes": n_antes, "ocultos_encontrados": n_ocultos,
                    "nulos_despues": df[col].isna().sum(),
                    "% nulos despues": round(df[col].isna().mean() * 100, 2)})
display(pd.DataFrame(resumen))

# Comprobación: los datos originales no se modificaron (los códigos ocultos siguen en df_original)
print("Códigos ocultos en admission_type_id -> df_original:", df_original["admission_type_id"].isin([5, 6, 8]).sum(),
      "| df (copia de trabajo):", df["admission_type_id"].isin([5, 6, 8]).sum())

Estos códigos se convirtieron a `NaN` para que reciban el **mismo tratamiento de faltantes** que el resto (sección *Faltantes* de la Etapa 4: moda en categóricas). Esto sube el porcentaje de faltantes de `admission_type_id`, `admission_source_id` y `discharge_disposition_id`.

In [ ]:
#patrones de datos faltantes
df["weight"].value_counts()

Weight tiene un 97% de valores faltantes. La ausencia es compatible con MAR si la tasa varía según errores de medición del hospital; no puede descartarse MNAR (por ejemplo, que se omita con más frecuencia en pacientes de mayor peso), pero no es verificable con estos datos. Dado el porcentaje, la variable se excluye del análisis

In [ ]:
msno.matrix(df)

In [ ]:
msno.bar(df, color='steelblue')

In [ ]:
# Serie auxiliar: no se agrega como columna a df (si no, se arrastra a df_imputado como una variable más)
weight_missing = df['weight'].isnull().rename('weight_missing')

df.groupby(weight_missing).mean(numeric_only=True)

##### Patrones de ausencia: supuesto MAR

Al revisar las gráficas de valores faltantes (matriz y barras) y comparar los registros con datos disponibles y faltantes, no identificamos patrones claros de ausencia. Por esta razón, **para este proyecto asumimos que los datos faltantes siguen un mecanismo MAR (Missing At Random)** y realizamos la imputación con esa consideración.

Esta decisión es una suposición de trabajo basada en la evidencia exploratoria disponible. No demuestra que el mecanismo sea MAR: con los datos observados no es posible descartar por completo MCAR o MNAR.


#### Valores fuera de rango

In [ ]:
df.describe()

 #### Distribuciones, categorías raras, desequilibrio de clases y posibles valores atípicos.

In [ ]:
for columna in df.select_dtypes(include='number').columns:
    sns.histplot(data=df, x=columna, bins=20)
    plt.title(f'Distribución de {columna}')
    plt.show()

In [ ]:
### 2. Categorías raras
for columna in df.select_dtypes(include='number').columns:
    print(df[columna].value_counts())

No se encontaron categorías raras

In [ ]:
#posibles atípicos
for columna in df.select_dtypes(include='number').columns:
    sns.boxplot(data=df, x=columna)
    plt.title(f'Boxplot de {columna}')
    plt.show()

Los boxplots sugieren la presencia de posibles valores atípicos en algunas variables numéricas. En esta fase únicamente se documentan; su tratamiento y validación se realizará durante la siguiente etapa del análisis.

#### Variables que podrían filtrar información futura o identificar directamente a una persona
Se identificaron variables como "encounter_id" y "patient_nbr". Estas variables no aportan información predictiva y podrían comprometer la privacidad de los registros, por lo que se consideran candidatas para excluirse en etapas posteriores- 

## Etapa 4. Limpieza, transformación y construcción de características: Módulo II

Justifiquen y documenten solo los procedimientos que correspondan a los datos:

- **Faltantes:** decidir si se conserva el registro, se imputa o se crea un indicador, explicar la elección
- **Atípicos y rangos:** distinguir un valor extremo válido de un error. Documentar reglas de detección y cualquier tratamiento, no eliminarlos automáticamente.
- **Escalas:** comparar unidades y distribuciones. Estandarización o normalización se ajustan en entrenamiento y se aplican sin recalcular a prueba.
- **Enriquecimiento:** crear variables derivadas que tengan sentido para la pregunta y explicar la fórmula.
- **Categóricas nominales:** considerar variables dummy/one-hot, no inventar un orden.
- **Categóricas ordinales:** definir y justificar el orden antes de codificar, las distancias numéricas implicadas también requieren justificación.
- **Variables polinomiales:** usarlas para representar curvatura cuando exista una razón.

Registren antes y después: tipos, dimensiones, faltantes y ejemplos de filas. Si no aplican una técnica, indiquen brevemente por qué no es pertinente.

# Faltantes y decisiones del equipo

En las variables se identificaron distintos porcentajes de datos faltantes, incluyendo vacíos ocultos convertidos a `NaN`. Para este proyecto asumimos un mecanismo **MAR (Missing At Random)** porque, al revisar las gráficas y comparar los registros con y sin datos, no observamos patrones claros de ausencia. Es una suposición de trabajo y no una demostración definitiva del mecanismo.

**Decisiones aplicadas:**
- `weight` (97%), `payer_code` (43%) y `medical_specialty` (52%) se eliminan por su alto porcentaje de faltantes.
- Las columnas numéricas discretas se imputan con la mediana calculada únicamente con el conjunto de entrenamiento.
- Las variables categóricas que se conservan se imputan con la moda calculada únicamente con entrenamiento.
- Los códigos administrativos sin información (`admission_type_id`, `admission_source_id` y `discharge_disposition_id`) se conservan como categoría `"Desconocido"` en lugar de rellenarlos con la moda.
- Se excluyen los ingresos cuyo `discharge_disposition_id` es 11, 13, 14, 19 o 20, correspondientes a altas por defunción o cuidados paliativos.
- Se conservan los ingresos repetidos, pero se separan los conjuntos agrupando por `patient_nbr`, para evitar que un mismo paciente aparezca tanto en entrenamiento como en prueba.

La división entrenamiento/prueba se realiza antes de imputar y escalar. Las transformaciones se ajustan con entrenamiento y después se aplican a prueba.


In [ ]:
# 1. Crear copia para no sobrescribir el DataFrame original
df_base = df.copy()

# 2. Excluir altas por defunción o cuidados paliativos
# Códigos señalados en la revisión: 11, 13, 14, 19 y 20.
codigos_no_reingreso = [11, 13, 14, 19, 20]
mascara_no_reingreso = df_base["discharge_disposition_id"].isin(codigos_no_reingreso)
print("Ingresos excluidos por defunción/cuidados paliativos:", int(mascara_no_reingreso.sum()))
df_base = df_base.loc[~mascara_no_reingreso].copy()

# 3. Eliminar variables con porcentaje alto de faltantes
excluir = ["weight", "payer_code", "medical_specialty"]
df_base = df_base.drop(columns=excluir)

# 4. Los códigos administrativos faltantes se conservan como categoría propia.
ids_administrativos = [
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id"
]
for col in ids_administrativos:
    df_base[col] = df_base[col].astype("object").where(df_base[col].notna(), "Desconocido")

# 5. Crear la variable objetivo antes de separar entrenamiento y prueba.
df_base["readmitted_30"] = (df_base["readmitted"] == "<30").astype(int)

# 6. Separar por paciente, no por fila: todos los ingresos de una persona
# quedan únicamente en entrenamiento o únicamente en prueba.
from sklearn.model_selection import GroupShuffleSplit

separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indices_train, indices_test = next(
    separador.split(df_base, groups=df_base["patient_nbr"])
)

train_idx = df_base.index[indices_train]
test_idx = df_base.index[indices_test]

df_train = df_base.loc[train_idx].copy()
df_test = df_base.loc[test_idx].copy()

print("Filas entrenamiento:", len(df_train))
print("Filas prueba:", len(df_test))
print("Pacientes compartidos entre conjuntos:",
      len(set(df_train["patient_nbr"]) & set(df_test["patient_nbr"])))

# 7. Imputar variables numéricas discretas con medianas calculadas SOLO en train.
cols_mediana = discretas.copy()

for col in cols_mediana:
    valor = df_train[col].median()
    df_train[col] = df_train[col].fillna(valor)
    df_test[col] = df_test[col].fillna(valor)

# 8. Imputar las variables categóricas restantes con la moda calculada SOLO en train.
cols_moda = [
    c for c in (nominales + ordinales)
    if c not in excluir + ids_administrativos
]

for col in cols_moda:
    moda_train = df_train[col].mode(dropna=True)
    if len(moda_train):
        valor = moda_train.iloc[0]
        df_train[col] = df_train[col].fillna(valor)
        df_test[col] = df_test[col].fillna(valor)

# 9. Reconstruir el DataFrame completo conservando los índices originales.
df_imputado = pd.concat([df_train, df_test]).sort_index()

print("Faltantes restantes:")
display(df_imputado.isna().sum()[df_imputado.isna().sum() > 0])


In [ ]:
# Eliminar columnas constantes e identificadores.
# encounter_id y patient_nbr se utilizaron para identificar ingresos y agrupar
# pacientes durante la división; no se usan como variables predictoras.
df_imputado = df_imputado.drop(
    columns=constantes + identificadores,
    errors="ignore"
)
df_imputado.isna().sum()[df_imputado.isna().sum() > 0]


# Enriquecimiento: agrupación de los códigos ICD-9

`diag_1`, `diag_2` y `diag_3` tienen cientos de códigos distintos (≈700 cada uno), demasiadas categorías para modelar o codificar con one-hot. Se agrupan en las **9 categorías de la Tabla 2 de Strack et al. (2014)**, que corresponden a capítulos de la CIE-9:

| Grupo | Códigos ICD-9 |
|---|---|
| Circulatory | 390–459, 785 |
| Respiratory | 460–519, 786 |
| Digestive | 520–579, 787 |
| Diabetes | 250.xx |
| Injury | 800–999 |
| Musculoskeletal | 710–739 |
| Genitourinary | 580–629, 788 |
| Neoplasms | 140–239 |
| Other | todo lo demás (incluye los códigos que empiezan con `V` y `E`) |

**Decisiones:**
- Los códigos `V` y `E` (factores que influyen en la salud y causas externas) no tienen rango numérico, así que caen en `Other`, igual que en el artículo ("other" para el resto).
- Se mantiene `Neoplasms` como grupo propio como en el artículo, aunque en este dataset (101,766 ingresos, sin filtrar un encuentro por paciente) pesa ≈3.2% (3.4% sin contar los imputados) y no ≈3.6%.
- Se parte de `df_imputado`, donde los faltantes de `diag_*` ya se rellenaron con la moda, según la decisión en *Faltantes*.

In [ ]:
def agrupar_icd9(codigo):
    """Agrupa un código ICD-9 en las categorías de la Tabla 2 (Strack et al., 2014)."""
    if pd.isna(codigo):
        return np.nan
    c = str(codigo).strip()
    if c.startswith(("E", "V")):      # códigos suplementarios sin rango numérico
        return "Other"
    try:
        n = float(c)
    except ValueError:
        return "Other"
    if 250 <= n < 251:                # 250.xx
        return "Diabetes"
    if 390 <= n < 460 or int(n) == 785:
        return "Circulatory"
    if 460 <= n < 520 or int(n) == 786:
        return "Respiratory"
    if 520 <= n < 580 or int(n) == 787:
        return "Digestive"
    if 580 <= n < 630 or int(n) == 788:
        return "Genitourinary"
    if 800 <= n < 1000:
        return "Injury"
    if 710 <= n < 740:
        return "Musculoskeletal"
    if 140 <= n < 240:
        return "Neoplasms"
    return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df_imputado[col + "_grupo"] = df_imputado[col].map(agrupar_icd9)

# Antes / después
print("Categorías antes :", {c: df_imputado[c].nunique() for c in ["diag_1", "diag_2", "diag_3"]})
print("Categorías después:", {c + "_grupo": df_imputado[c + "_grupo"].nunique() for c in ["diag_1", "diag_2", "diag_3"]})
print("Nulos en grupos:", df_imputado[["diag_1_grupo", "diag_2_grupo", "diag_3_grupo"]].isna().sum().to_dict())

tabla_icd = (df_imputado["diag_1_grupo"].value_counts().rename("n_ingresos").to_frame()
             .assign(pct=lambda t: (t["n_ingresos"] / t["n_ingresos"].sum() * 100).round(1)))
display(tabla_icd)
display(df_imputado[["diag_1", "diag_1_grupo", "diag_2", "diag_2_grupo"]].sample(8, random_state=42))

In [ ]:
# Tasa de readmisión <30 días por grupo de diagnóstico principal (revisión de sentido clínico)
(df_imputado.assign(readm30=df_imputado["readmitted"].eq("<30"))
 .groupby("diag_1_grupo")["readm30"].agg(["mean", "size"]).round(3)
 .sort_values("mean", ascending=False))

In [ ]:
# Los códigos originales (~700 categorías) se reemplazan por los grupos en el dataset de trabajo;
# el dataframe `df` conserva los códigos originales.
df_imputado = df_imputado.drop(columns=["diag_1", "diag_2", "diag_3"])
nominales_modelo = [c for c in df_imputado.columns if c.endswith("_grupo")]
print(df_imputado.shape)

# Enriquecimiento: variable objetivo binaria

La pregunta del proyecto es si el paciente reingresa **en menos de 30 días**. `readmitted` tiene tres valores (`NO`, `>30`, `<30`), así que se crea `readmitted_30`:

- **evento = 1:** reingreso en menos de 30 días (`<30`)
- **no evento = 0:** reingreso después de 30 días (`>30`) o ningún reingreso (`NO`)

`readmitted` y `readmitted_30` son la misma información: **ninguna de las dos puede usarse como predictora**. Esta variable se necesita para WOE/IV y LDA (Etapa 5).

In [ ]:
print(df_imputado["readmitted_30"].value_counts().rename(
    {0: "No evento (0)", 1: "Reingreso <30 días (1)"}
))
print(f"Proporción del evento: {df_imputado['readmitted_30'].mean() * 100:.1f}%  (clase minoritaria)")
display(pd.crosstab(df_imputado["readmitted"], df_imputado["readmitted_30"]))

# Recuperar la variable objetivo de cada partición.
y_train = df_imputado.loc[train_idx, "readmitted_30"]
y_test = df_imputado.loc[test_idx, "readmitted_30"]


# Atípicos y rangos:
Distinguir un valor extremo válido de un error. Documentar reglas de detección y cualquier tratamiento, no eliminarlos automáticamente.

In [ ]:
cols_numericas = [c for c in discretas if c not in ["encounter_id", "patient_nbr"]]

resumen_outliers = []

for col in cols_numericas:
    Q1 = df_imputado[col].quantile(0.25)
    Q3 = df_imputado[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    n_outliers = ((df_imputado[col] < lim_inf) | (df_imputado[col] > lim_sup)).sum()
    pct_outliers = (n_outliers / len(df_imputado)) * 100

    resumen_outliers.append({
        "Variable": col,
        "Min": df_imputado[col].min(),
        "Max": df_imputado[col].max(),
        "Lim_Sup_IQR": round(lim_sup, 2),
        "Outliers": n_outliers,
        "% Outliers": round(pct_outliers, 2)
    })
df_outliers = pd.DataFrame(resumen_outliers)
display(df_outliers)

time_in_hospital se detectaron 2% de valores atípicos pero estàn en el rango aceptado por nuestra informacion (1-14 dias) por lo tanto los dejamos como una ocurrencia natural del dataset

num_lab_procedures No creemos que sean errores por la cantidad de estos fuera del rango.

num_procedures	Estan en un rango que consideramos normal (1-6) no hay un dato que salga de ese rango por mucho, entonces los mantenemos

num_medications No tenemos tantas medicinas registradas para el tratamiento de la diabetes, podrian ser medicinas utilizadas en un historico por el paciente, sin ser exclusivamente para la diabetes.

number_outpatient Varia mucho dependiendo de la salud general del paciente, es normal que existan valores atipicos

number_emergency Varia mucho dependiendo de la salud general del paciente, es normal que existan valores atipicos

number_impatient Varia mucho dependiendo de la salud general del paciente, es normal que existan valores atipicos

** number_diagnoses Investigar bien que significa, podrian ser el numero de enfermedades/condiciones medicas detectadas

### Estandarización o normalización

In [ ]:
# Primero se utiliza Min-Max para llevar las variables numéricas al rango [1, 2].
# Se ajusta SOLO con entrenamiento y se aplica la misma transformación a prueba.
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, RobustScaler

minmax = MinMaxScaler(feature_range=(1, 2), clip=True)

X_train_mm = pd.DataFrame(
    minmax.fit_transform(df_imputado.loc[train_idx, discretas]),
    index=train_idx, columns=discretas
)
X_test_mm = pd.DataFrame(
    minmax.transform(df_imputado.loc[test_idx, discretas]),
    index=test_idx, columns=discretas
)

# Sustituir en el DataFrame los valores ya normalizados.
df_imputado.loc[train_idx, discretas] = X_train_mm
df_imputado.loc[test_idx, discretas] = X_test_mm

display(X_train_mm.head())


In [ ]:
df_imputado[discretas].skew()

In [ ]:
# Box-Cox para reducir la asimetría después de Min-Max.
# PowerTransformer estima los parámetros lambda SOLO con entrenamiento.
boxcox = PowerTransformer(method="box-cox", standardize=False)

X_train_boxcox = pd.DataFrame(
    boxcox.fit_transform(df_imputado.loc[train_idx, discretas]),
    index=train_idx, columns=discretas
)
X_test_boxcox = pd.DataFrame(
    boxcox.transform(df_imputado.loc[test_idx, discretas]),
    index=test_idx, columns=discretas
)

df_imputado.loc[train_idx, discretas] = X_train_boxcox
df_imputado.loc[test_idx, discretas] = X_test_boxcox

display(X_train_boxcox.head())


In [ ]:
df_imputado[discretas].skew()

In [ ]:
import matplotlib.pyplot as plt

for col in discretas:
    plt.figure(figsize=(6, 4))
    plt.hist(df_imputado[col], bins=30, edgecolor='black')
    plt.title(f'Distribución de {col}')
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

for col in discretas:
    plt.figure(figsize=(6, 4))
    plt.boxplot(df_imputado[col].dropna())
    plt.title(f'Boxplot de {col}')
    plt.ylabel(col)
    plt.show()

Se nota que disminuyó la asimetría de los datos, sin embargo aun esta la presencia de datos atípicos por tanto se aplica una normalización robusta

In [ ]:
# RobustScaler: también se ajusta solo con entrenamiento.
# Se aplica a ambos conjuntos usando la mediana y el rango intercuartílico de train.
scaler = RobustScaler()

X_train_robust = pd.DataFrame(
    scaler.fit_transform(df_imputado.loc[train_idx, discretas]),
    index=train_idx, columns=discretas
)
X_test_robust = pd.DataFrame(
    scaler.transform(df_imputado.loc[test_idx, discretas]),
    index=test_idx, columns=discretas
)

df_imputado.loc[train_idx, discretas] = X_train_robust
df_imputado.loc[test_idx, discretas] = X_test_robust

display(df_imputado[discretas].describe())


### Categóricas nominales: 
considerar variables dummy/one-hot, no inventar un orden.

Cada medicamento tiene cuatro posibles categorías:

Down  |	Disminuyó la dosis del medicamento

No | No se utilizó el medicamento

Steady	| La dosis se mantuvo estable

Up |	Aumentó la dosis del medicamento.

Se usará One-Hot encoding porque estas variables son nominales.

La categoría de referencia es `No` (no se usó el medicamento), por lo que se elimina la columna `<medicamento>_No`. (Con `drop_first=True` se habría eliminado `Down`, la categoría más rara, por orden alfabético.)

In [ ]:
df_largo = df_imputado.reset_index().melt(
    id_vars='index',
    value_vars=medicamentos,
    var_name='variable',
    value_name='estado'
)

df_largo = pd.concat([
    df_largo,
    pd.get_dummies(df_largo['estado'], dtype=int)
], axis=1)

df_largo = df_largo.drop(columns='estado')
df_largo

In [ ]:
df_imputado.head()

## Categóricas ordinales:
Se define y justifica el orden **antes** de codificar:

| Variable | Orden (de menor a mayor) | Justificación |
|---|---|---|
| `age` | [0-10) < [10-20) < … < [90-100) | Orden natural de los intervalos de edad |
| `max_glu_serum` | Norm < >200 < >300 | Niveles crecientes de glucosa en suero |
| `A1Cresult` | Norm < >7 < >8 | Niveles crecientes de HbA1c |

**`None` en `max_glu_serum` y `A1Cresult` significa "prueba no realizada".** No es un nivel más alto ni más bajo que los demás, por lo que no se coloca dentro de la escala. Se maneja con un indicador `<variable>_medido` (1 = se midió, 0 = no se midió) y recibe 0 en los contrastes, para que el modelo distinga "no medido" sin mezclarlo con la gravedad. En el artículo, "no medido" también es una categoría aparte de HbA1c.

**Contrastes polinómicos ortogonales** (equivalentes a `contr.poly` de R): una columna lineal (tendencia) y una cuadrática (curvatura).
- Suponen niveles equiespaciados. Es razonable en `age` (intervalos de 10 años) y discutible en glucosa y HbA1c, donde los cortes no son equidistantes; es una simplificación que debe reportarse.
- La componente cuadrática en `age` se justifica con la Figura 2 del artículo: la readmisión sube rápido hasta los ~30 años y después casi se aplana.

In [ ]:
def contrastes_polinomicos(niveles, grado=2):
    """Contrastes polinómicos ortogonales (como contr.poly de R) para niveles ordenados y equiespaciados.
    Devuelve un DataFrame con una fila por nivel: columna 'lin' (tendencia) y 'cuad' (curvatura)."""
    k = len(niveles)
    x = np.arange(k) - (k - 1) / 2
    V = np.vander(x, grado + 1, increasing=True)      # columnas: 1, x, x^2
    Q, _ = np.linalg.qr(V)                            # ortogonalización
    Q = Q[:, 1:]                                      # se quita la constante
    Q = Q * np.sign(Q[-1, :])                         # cada componente es positiva en el último nivel
    return pd.DataFrame(Q, index=niveles, columns=["lin", "cuad", "cub"][:grado])

orden_ordinales = {
    "age": ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
            "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"],
    "max_glu_serum": ["Norm", ">200", ">300"],      # "None" = no medido (fuera de la escala)
    "A1Cresult": ["Norm", ">7", ">8"],              # "None" = no medido (fuera de la escala)
}

In [ ]:
for col, orden in orden_ordinales.items():
    # Validación: no debe haber categorías fuera del orden definido
    extra = set(df_imputado[col].unique()) - set(orden) - {"None"}
    assert not extra, f"{col}: categorías sin orden definido {extra}"

    contr = contrastes_polinomicos(orden)
    if col != "age":
        df_imputado[col + "_medido"] = (df_imputado[col] != "None").astype(int)
    for comp in contr.columns:
        # "None" (no medido) no está en contr -> 0 en el contraste
        df_imputado[f"{col}_{comp}"] = df_imputado[col].map(contr[comp]).fillna(0.0)

for col, orden in orden_ordinales.items():
    print(col)
    display(contrastes_polinomicos(orden).round(3))

df_imputado[["age", "age_lin", "age_cuad", "A1Cresult", "A1Cresult_medido", "A1Cresult_lin", "A1Cresult_cuad"]].head()

In [ ]:
# Correspondencia entre cada nivel y su contraste (distinto estilo de línea/marcador para escala de grises)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (col, orden) in zip(axes, orden_ordinales.items()):
    contr = contrastes_polinomicos(orden)
    ax.plot(range(len(orden)), contr["lin"], marker="o", linestyle="-", label="lineal")
    ax.plot(range(len(orden)), contr["cuad"], marker="s", linestyle="--", label="cuadrático")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_xticks(range(len(orden)))
    ax.set_xticklabels(orden, rotation=45)
    ax.set_title(f"Contrastes de {col}")
    ax.set_ylabel("Coeficiente")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# One-hot de medicamentos con "No" como categoría de referencia.
# Se generan las columnas a partir de entrenamiento y se alinean las de prueba.
df_train_dum = pd.get_dummies(
    df_imputado.loc[train_idx], columns=medicamentos, dtype=int
)
df_test_dum = pd.get_dummies(
    df_imputado.loc[test_idx], columns=medicamentos, dtype=int
)

# Prueba utiliza exactamente las columnas aprendidas en entrenamiento.
df_test_dum = df_test_dum.reindex(columns=df_train_dum.columns, fill_value=0)

# Quitar "No" como categoría de referencia.
cols_no = [f"{m}_No" for m in medicamentos]
df_train_dum = df_train_dum.drop(columns=cols_no, errors="ignore")
df_test_dum = df_test_dum.drop(columns=cols_no, errors="ignore")

df_dum = pd.concat([df_train_dum, df_test_dum]).sort_index()

# Matrices finales separadas. Se excluyen las dos versiones de la variable objetivo.
columnas_objetivo = ["readmitted", "readmitted_30"]
X_train = df_dum.loc[train_idx].drop(columns=columnas_objetivo, errors="ignore")
X_test = df_dum.loc[test_idx].drop(columns=columnas_objetivo, errors="ignore")
y_train = df_dum.loc[train_idx, "readmitted_30"]
y_test = df_dum.loc[test_idx, "readmitted_30"]

print("Dimensiones X_train:", X_train.shape)
print("Dimensiones X_test:", X_test.shape)
print("Dimensiones y_train / y_test:", y_train.shape, y_test.shape)
df_dum.head()


## Etapa 5. Selección y transformación de características: 

### 5.1 Selección por varianza

Calcular varianza en las variables numéricas pertinentes e identificar constantes o casi constantes. La varianza depende de las unidades: una columna en metros y la misma columna en milímetros tienen varianzas muy distintas. Un indicador poco frecuente puede ser predictivo aunque su varianza sea baja

### 5.2 Selección por correlación

Mostrar una matriz o mapa de calor para las características relevantes. Elegir Pearson para asociación lineal o Spearman para asociación monótona por rangos, y justificarlo. Si filtran pares altamente correlacionados, declarar un umbral (por ejemplo, 0.85) y una regla reproducible para decidir cuál conservar. Reportar las columnas retiradas y explicar el criterio.

La correlación mide asociación, no causalidad ni independencia. Correlación baja no descarta dependencia no lineal, un umbral por pares tampoco encuentra redundancia multivariable. 

### 5.3 WOE e Information Value

Usar WOE/IV solo si existe una variable objetivo **binaria** y la pregunta justifica una codificación supervisada. Definir explícitamente evento=1 y no evento=0. Para cada predictor mostrar la tabla de grupos, conteos por clase, distribuciones, WOE por grupo e IV total.

Para variables continuas, justificar bins y revisar tamaños, faltantes y conteos cero. Para variables categóricas, decidir qué categorías se agrupan y por qué. 


### 5.4 Análisis discriminante lineal (LDA)

LDA es supervisado: usa las etiquetas para buscar combinaciones que separen clases. Si es pertinente, documentar escalado, variables de entrada, priors y número de componentes. El máximo de ejes discriminantes es $\min(p, C-1)$, donde $p$ es el número de características y $C$ el número de clases. Con dos clases solo hay un eje discriminante.

Distinguir LDA como transformador para visualizar/reducir dimensiones de LDA como clasificador. Mostrar una proyección 1D o 2D cuando sea posible y explicar qué clases se separan. 

### 5.5 Análisis de componentes principales (PCA)

PCA es no supervisado: encuentra combinaciones que conservan varianza, sin consultar la etiqueta. Estandarizar las variables cuando sus escalas/unidades sean distintas y justificar el escalado cuando deban conservarse magnitudes originales. El ajuste y el escalador se aprenden solo en entrenamiento.

Mostrar varianza explicada individual y acumulada, justificar cuántos componentes se conservan. Interpretar los pesos principales y, si ayuda, graficar las primeras componentes. 

# Entregables obligatorios

Cada equipo entregará estos tres productos:

### 1. Presentación con diapositivas

**(15 min de presentación y 5 min de preguntas):** Debe comunicar el problema, los datos asignados, el método de procesamiento, los hallazgos más importantes, las limitaciones y las conclusiones. Incluyan figuras legibles, títulos y una explicación breve de qué evidencia aporta cada una. Todas las personas del equipo deben poder explicar el trabajo y responder preguntas.

### 2. Reporte digital del trabajo realizado

El **reporte** se entregará como documento digital elaborado en computadora; **no se aceptan fotografías de hojas o cuadernos**. Como mínimo, debe incluir:

- Nombre y apellidos de todas las personas integrantes.
- Descripción del problema, pregunta, objetivo y solución propuesta.
- Fuente, liga del dataset, descripción de datos y diccionario de variables.
- **Proceso realizado (Etapa 1-5)**: extracción, limpieza, decisiones de transformación y métodos utilizados.
- Resultados con figuras/tablas y su interpretación.
- **Un capítulo que describa la solución que se propuso para el problema.**
- **El código generado por el equipo para realizar el proyecto.** Puede organizarse en bloques legibles en el reporte y complementarse con el notebook entregado por separado, debe quedar claro qué código produjo los resultados.
- **Una sección de conclusiones.**
- Limitaciones, referencias y explicación de las decisiones relevantes.

**Impresión:** no es necesario imprimir a color. Diseñen las figuras para que se distingan en escala de grises, usen tipos de línea, marcadores, etiquetas o patrones además del color.

**Estructura sugerida del reporte:** 
- portada e integrantes
- resumen
- contexto de los datos y fuente
- Metodología/proceso
- Capítulo de solución propuesta (Etapas 1- 5)
- Resultados y figuras
- Limitaciones
- Conclusiones, referencias
- Apéndice de código si se requiere.

### 3. Código en Jupyter notebook

El notebook debe documentar el proceso completo: código, análisis, comentarios, tablas y gráficas. Debe ejecutarse en orden y ser comprensible para alguien que no participó en su elaboración.

- Incluir nombres completos, equipo, pregunta y liga asignada.
- Mantener comentarios y texto Markdown que expliquen qué se hace y por qué.
- No omitir celdas necesarias ni depender de variables creadas manualmente en otra sesión.
- Usar rutas relativas, indicar formato y fuentes de datos y no incluir credenciales privadas.
- Mostrar las figuras y salidas que se discuten en el reporte.
- Registrar decisiones, parámetros y semilla aleatoria cuando aplique.
- Ejecutar desde un kernel limpio antes de entregar y revisar que no haya errores.
- Mantener la versión original de los datos; no guardar archivos derivados sobre el original sin indicarlo.